# NLP-Based Fake Job Post Detector - model training notebook
Run the cells from top to bottom. The notebook lives in `notebooks/`, so the first cell moves to the project folder.

In [ ]:
import os, sys
os.chdir('..')            # go from notebooks/ to the project root
sys.path.append(os.getcwd())
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import clean_text, explain_steps, TARGET_COLUMN

## 1. Load the dataset
Put `fake_job_postings.csv` in the `data/` folder first (see README).

In [ ]:
df = pd.read_csv('data/fake_job_postings.csv')
print(df.shape)
df.head()

## 2. Explore: target variable and missing values
`fraudulent` is the target (1 = fake, 0 = real).

In [ ]:
print(df[TARGET_COLUMN].value_counts())
sns.countplot(x=TARGET_COLUMN, data=df)
plt.title('Real (0) vs Fake (1)')
plt.show()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## 3. See the NLP preprocessing on one job post

In [ ]:
sample = str(df.loc[0, 'description'])
for stage, value in explain_steps(sample).items():
    print(stage)
    print(value if isinstance(value, str) else ' '.join(value[:40]))
    print()

## 4. Train and compare the four models
This runs the same code as `python -m src.train_model` (split, TF-IDF, 4 models, evaluation, saving).

In [ ]:
from src.train_model import run_training
results = run_training('data/fake_job_postings.csv')
results

## 5. Look at the saved charts

In [ ]:
from IPython.display import Image, display
for name in ['model_comparison.png', 'confusion_matrices.png']:
    display(Image('reports/' + name))

## 6. Test the saved model on the demo advertisements

In [ ]:
from src.predict import JobPostDetector
from src.sample_jobs import SAMPLE_JOBS
detector = JobPostDetector()
for job in SAMPLE_JOBS:
    r = detector.predict(job)
    print(job['name'], '->', r['label'], f"(fake score {r['fake_probability']:.2f})")